In [8]:
# Imports 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

In [9]:
# Load Data 

df = pd.read_csv("../data.csv")
df.head()

,"PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked"
0,"1,0,3,""Braund, Mr. Owen Harris"",male,22,1,0,A/..."
1,"2,1,1,""Cumings, Mrs. John Bradley (Florence Br..."
2,"3,1,3,""Heikkinen, Miss. Laina"",female,26,0,0,S..."
3,"4,1,1,""Futrelle, Mrs. Jacques Heath (Lily May ..."
4,"5,0,3,""Allen, Mr. William Henry"",male,35,0,0,3..."


In [11]:
#EdA

df.info()
df.describe()

plt.figure(figsize=(5,4))
sns.countplot(x="Survived", data=df)
plt.show()

sns.countplot(x="Sex", hue="Survived", data=df)
plt.show()

sns.countplot(x="Pclass", hue="Survived", data=df)
plt.show()

sns.heatmap(df.isnull(), cbar=False)
plt.show()

df.corr(numeric_only=True)
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap="coolwarm")
plt.show()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 1 columns):
 #   Column                                                                           Non-Null Count  Dtype
---  ------                                                                           --------------  -----
 0   PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked  891 non-null    str  
dtypes: str(1)
memory usage: 7.1 KB


ValueError: Could not interpret value `Survived` for `x`. An entry with this name does not appear in `data`.

<Figure size 500x400 with 0 Axes>

In [ ]:
# Cleaning

df['Age'] = df['Age'].fillna(df['Age'].median())
df["Embarked"] = df["Embarked"].fillna(df['Embarked'].mode()[0])
df = df.drop(columns=['Cabin', 'Ticket'])
df = df.drop_duplicated()

In [ ]:
# Encoding 

df['Sex'] = df['Sex'].map({'male':0, 'female':1})
df['Embarked'] = df['Embarked'].map({'S':0,'C':1,'Q':2})

df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.')

title_map = {
    "Mr":"Mr", "Mrs":"Mrs", "Miss":"Miss", "Master":"Master",
    "Mlle":"Miss","Ms":"Miss","Mme":"Mrs"
}

df['Title'] = df['Title'].map(title_map)
df['Title'] = df['Title'].fillna("Other")

In [ ]:
# Feature Engineering


df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

df['AgeBin'] = pd.cut(df['Age'], bins=[0,12,20,40,120], labels=[0,1,2,3])
df['FareBin'] = pd.qcut(df['Fare'], 4, labels=[0,1,2,3])

df = df.drop(columns=['Name', 'PassengerId'])

df = df.dropna()


In [ ]:
# Train / Test split



X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# baseline model 

model_lr = LogisticRegression(max_iter=2000)
model_lr.fit(X_train, y_train)
pred_lr = model_lr.predict(X_test)

print("LR Accuracy:", accuracy_score(y_test, pred_lr))
print(confusion_matrix(y_test, pred_lr))
print(classification_report(y_test, pred_lr))


In [ ]:
# Better Model ----- Random Forest


model_rf = RandomForestClassifier()
modeit(X_train, y_train)
pred_rf = model_rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, pred_rf))
print(confusion_matrix(y_test, pred_rf))
print(classification_report(y_test, pred_rf))
l_rf.f

In [ ]:
# HyperParameter Tuning


params = {
    'n_estimators':[100,200,300],
    'max_depth':[4,6,8],
    'min_samples_split':[2,5,10]
}

grid = GridSearchCV(RandomForestClassifier(), params, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
pred_best = best_model.predict(X_test)

print("BEST MODEL ACCURACY:", accuracy_score(y_test, pred_best))
print(confusion_matrix(y_test, pred_best))
print(classification_report(y_test, pred_best))
print("Best Params:", grid.best_params_)

In [ ]:
# Features Importance

importance = pd.Series(best_model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).plot(kind="bar", figsize=(10,4))
plt.show()


In [ ]:
# Final Inference pipeline

joblib.dump(best_model, "titanic_model.pkl")

loaded = joblib.load("titanic_model.pkl")
sample = X_test.iloc[:5]

print("Predictions from saved model:", loaded.predict(sample))